# Ramsey ???Notebook ???

??????????????????????P1 ??????????????????????

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from qsim.ui.notebook import run_workflow, plot_default

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'examples' / 'backend.yaml').exists():
            return p
    raise FileNotFoundError('Cannot locate project root with pyproject.toml and examples/backend.yaml')

PROJECT_ROOT = find_project_root(Path.cwd())
BACKEND_PATH = (PROJECT_ROOT / 'examples' / 'backend.yaml').resolve()
OUT_ROOT = (PROJECT_ROOT / 'runs' / 'notebook' / 'ramsey').resolve()
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('BACKEND_PATH =', BACKEND_PATH)
print('OUT_ROOT =', OUT_ROOT)

In [ ]:
def make_ramsey_qasm(idle_cycles: int) -> str:
    idle_lines = '\n'.join(['id q[0];' for _ in range(int(idle_cycles))])
    return f'''
OPENQASM 3;
qubit[1] q;
bit[1] c;
sx q[0];
{idle_lines}
sx q[0];
measure q[0] -> c[0];
'''

print('Quantum Circuit (QASM, idle=8):')
print(make_ramsey_qasm(8))

In [ ]:
idle_cycles = np.arange(0, 80, 2)
final_p1 = []
delay_times = []
sample_result = None

hw = {
    'gate_duration': 3.0,
    'dt': 0.1,
    'qubit_freqs_hz': [0.03],
    'control_scale': 0.22,
}
noise = {'gamma1': 0.0, 'gamma_phi': 0.01}

for i, n_idle in enumerate(idle_cycles):
    qasm = make_ramsey_qasm(int(n_idle))
    result = run_workflow(
        qasm_text=qasm,
        backend_path=str(BACKEND_PATH),
        out_dir=str((OUT_ROOT / f'idle_{int(n_idle):03d}').resolve()),
        hardware=hw,
        noise=noise,
        engine='qutip',
    )
    final_p1.append(result['trace'].states[-1][0])
    delay_times.append(float(n_idle) * hw['gate_duration'])
    if i == len(idle_cycles) // 2:
        sample_result = result

final_p1 = np.asarray(final_p1)
delay_times = np.asarray(delay_times)

# ??????????Ramsey ?????
detune = float(hw['qubit_freqs_hz'][0])
gamma_phi = float(noise['gamma_phi'])
p1_theory = 0.5 * (1.0 + np.exp(-gamma_phi * delay_times) * np.cos(2.0 * np.pi * detune * delay_times))

print('Last run dir =', result['out_dir'])

In [ ]:
# ?? 1: ???? + ????
plt.figure(figsize=(8, 4))
plt.plot(delay_times, final_p1, 'o-', lw=1.5, label='simulation readout P1')
plt.plot(delay_times, p1_theory, '--', lw=2.0, label='theory (damped cosine)')
plt.xlabel('idle time (arb. units)')
plt.ylabel('final excited population P1')
plt.title('Ramsey Interference')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print('P1 min/max =', float(final_p1.min()), float(final_p1.max()))

In [ ]:
# ?? 2: ???????????????
figs = plot_default(sample_result)
display(figs['pulses'])

In [ ]:
# ?? 3: ???????????readout time trace?
trace = sample_result['trace']
trace_p1 = np.asarray([row[0] for row in trace.states], dtype=float)

plt.figure(figsize=(8, 3.5))
plt.plot(trace.times, trace_p1, lw=1.6)
plt.xlabel('time')
plt.ylabel('P1(t)')
plt.title('Single-run readout trace')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import json

print('--- Settings Report ---')
print(json.dumps(result['settings'], ensure_ascii=False, indent=2))
print('--- Timings (s) ---')
for k, v in sorted(result['timings'].items(), key=lambda x: x[1], reverse=True):
    print(f'{k:16s} {v:.6f}')
